# Manejo de autenticación con JWT

In [ ]:
# Ajuste de modelo user, para que sea utilizado como modelo principal de usuario de django
from django.db import models
from django.utils import timezone
from django.contrib.auth.models import AbstractBaseUser, PermissionsMixin, BaseUserManager

GENDER = [
    ("female", "Femenino"),
    ("male", "Masculino"),
    ("other", "Otro"),
]

class UserManager(BaseUserManager):
    def create_user(self, email, name, paternal_name, maternal_name, age, gender, password=None, **extra_fields):
        if not email:
            raise ValueError("El usuario debe tener un email")
        email = self.normalize_email(email)
        user = self.model(
            email=email,
            name=name,
            paternal_name=paternal_name,
            maternal_name=maternal_name,
            age=age,
            gender=gender,
            **extra_fields
        )
        user.set_password(password)
        user.save(using=self._db)
        return user

    def create_superuser(self, email, name, paternal_name, maternal_name, age, gender, password=None, **extra_fields):
        extra_fields.setdefault("is_staff", True)
        extra_fields.setdefault("is_superuser", True)
        return self.create_user(email, name, paternal_name, maternal_name, age, gender, password, **extra_fields)

class User(AbstractBaseUser, PermissionsMixin):
    name            = models.CharField(max_length=100)
    paternal_name   = models.CharField(max_length=100)
    maternal_name   = models.CharField(max_length=100, blank=True, null=True, default="")
    age             = models.IntegerField()
    email           = models.EmailField(unique=True)
    gender          = models.CharField(max_length=10, choices=GENDER)
    phone           = models.CharField(max_length=20, blank=True, null=True)

    created_at      = models.DateTimeField(default=timezone.now)
    updated_at      = models.DateTimeField(auto_now=True)

    is_active       = models.BooleanField(default=True)
    is_staff        = models.BooleanField(default=False)

    objects         = UserManager()

    USERNAME_FIELD = "email"
    REQUIRED_FIELDS = ["name", "paternal_name", "maternal_name", "age", "gender"]

    def __str__(self):
        return self.email


In [ ]:
# Ajustes en serialización general
# src\users\serializers.py

from rest_framework import serializers

from users.models import User

class UserSerializer(serializers.ModelSerializer):
    password = serializers.CharField(write_only=True)

    class Meta:
        model = User
        fields = [
            "id",
            "email",
            "password",
            "name",
            "paternal_name",
            "maternal_name",
            "age",
            "gender",
            "phone",
            "created_at",
            "updated_at",
        ]
        read_only_fields = ["id", "created_at", "updated_at"]

    def create(self, validated_data):
        password = validated_data.pop("password", None)
        user = User(**validated_data)
        if password:
            user.set_password(password)
        user.save()
        return user

    def update(self, instance, validated_data):
        password = validated_data.pop("password", None)
        for attr, value in validated_data.items():
            setattr(instance, attr, value)
        if password:
            instance.set_password(password)
        instance.save()
        return instance


In [ ]:
# Creación de vista para GET y POST de usuario
# src\users\views.py

from django.shortcuts import render
from .forms import UserForm

from rest_framework.response import Response
from rest_framework import status
from rest_framework.viewsets import ViewSet
from .serializers import UserSerializer
from .models import User
from rest_framework import status
from rest_framework.views import APIView
from rest_framework_simplejwt.tokens import RefreshToken

from users.models import User
from rest_framework.permissions import IsAuthenticated


...


class RegisterView(APIView):
    def post(self, request):
        serializer = UserSerializer(data=request.data)
        if serializer.is_valid():
            user = serializer.save()
            refresh = RefreshToken.for_user(user)
            return Response({
                "user": UserSerializer(user).data,
                "refresh": str(refresh),
                "access": str(refresh.access_token),
            }, status=status.HTTP_201_CREATED)
        return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

    def get(self, request):
        """Devuelve los datos del usuario autenticado (requiere JWT válido)"""
        if not request.user or not request.user.is_authenticated:
            return Response(
                {"error": "No autenticado"},
                status=status.HTTP_401_UNAUTHORIZED
            )
        serializer = UserSerializer(request.user)
        return Response(serializer.data)

    def get_permissions(self):
        if self.request.method == "GET":
            return [IsAuthenticated()]
        return []

In [ ]:
# Ajuste de URLS del módulo users
# src\users\urls.py
from django.urls import path, include
from .views import userCreate, APIUser
from rest_framework.routers import DefaultRouter
from rest_framework_simplejwt.views import (
    TokenObtainPairView,
    TokenRefreshView,
)
from .views import RegisterView

router = DefaultRouter()
router.register("api-user",APIUser, basename="api-user")



urlpatterns = [
    path("create/",userCreate),
    path("",include(router.urls)),
    path("api/register/", RegisterView.as_view(), name="register"),
    path("api/token/", TokenObtainPairView.as_view(), name="token_obtain_pair"),
    path("api/token/refresh/", TokenRefreshView.as_view(), name="token_refresh"),
]


In [ ]:
# Configuración general del proyecto
# src\config\settings.py

# Application definitions
INSTALLED_APPS = [
    "user_app.apps.UserAppConfig",
    "accounts.apps.AccountsConfig",
    "addresses.apps.AddressesConfig",
    "analytics.apps.AnalyticsConfig",
    "billing.apps.BillingConfig",
    "carts.apps.CartsConfig",
    "products.apps.ProductsConfig",
    "order.apps.OrderConfig",
    "pages.apps.PagesConfig",
    "users.apps.UsersConfig",
    "django.contrib.admin",
    "django.contrib.sessions",
    "django.contrib.messages",
    "django.contrib.staticfiles",
    "django.contrib.auth",
    "django.contrib.contenttypes",
    "rest_framework",
    'rest_framework.authtoken'
]

...

REST_FRAMEWORK = {
    'DEFAULT_RENDERER_CLASSES': [
        'rest_framework.renderers.JSONRenderer',
    ],
    'DEFAULT_AUTHENTICATION_CLASSES': [
        'rest_framework_simplejwt.authentication.JWTAuthentication',
    ]
}

SIMPLE_JWT = {
    "ROTATE_REFRESH_TOKENS":True,
}

AUTH_USER_MODEL = "users.User"
